In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
import time
from nba_api.stats.endpoints import shotchartdetail, leaguedashplayerstats, leaguegamefinder, playbyplayv3
import numpy as np
import duckdb
import isodate
from IPython.display import display
import json


# Load Data for Shots

Defines a function to download season shotchart data and selected player skill stats, merges them per season, optimizes dtypes, and saves the combined dataset to Parquet.

In [ ]:
# -----------------------------
# Konfiguration
# -----------------------------
CUSTOM_HEADERS = {
    'Host': 'stats.nba.com',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:91.0) Gecko/20100101 Firefox/91.0',
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'en-US,en;q=0.5',
    'Referer': 'https://www.nba.com/',
    'Connection': 'keep-alive',
}

def create_comprehensive_nba_dataset(start_year=2019, end_year=2025, filename="nba_raw_shot_data.parquet"):
    """
    Lädt Wurfdaten und Spieler-Stats (Skill) pro Saison, merged sie und speichert als Parquet.
    
    Args:
        start_year (int): Startjahr (z.B. 2019 für Saison 2019-20)
        end_year (int): Endjahr (exklusiv, z.B. 2025)
        filename (str): Name der Output-Datei
    """
    
    all_seasons_data = []
    seasons = [f"{y}-{str(y+1)[-2:]}" for y in range(start_year, end_year)]

    for season in seasons:
        try:
            # 1. Wurfdaten holen
            shot_data = shotchartdetail.ShotChartDetail(
                player_id=0,
                team_id=0,
                season_nullable=season,
                context_measure_simple="FGA",
                headers=CUSTOM_HEADERS,
                timeout=120
            ).get_data_frames()[0]
            
            # 2. Spieler-Skill-Daten für DIESE Saison holen
            player_stats = leaguedashplayerstats.LeagueDashPlayerStats(
                season=season,
                headers=CUSTOM_HEADERS,
                timeout=120
            ).get_data_frames()[0]

            # 3. Relevante Skill-Spalten auswählen und umbenennen
            # Wir wollen nicht alles, nur was für xPTS relevant ist
            skill_cols = player_stats[['PLAYER_ID', 'FG_PCT', 'FG3_PCT', 'FT_PCT']]
            skill_cols = skill_cols.rename(columns={
                'FG_PCT': 'PLAYER_SEASON_FG_PCT',   # Umbenennen um Verwechslung zu vermeiden
                'FG3_PCT': 'PLAYER_SEASON_3P_PCT',
                'FT_PCT': 'PLAYER_SEASON_FT_PCT'
            })

            # 4. Daten zusammenfügen (Merge)
            merged_df = pd.merge(shot_data, skill_cols, on='PLAYER_ID', how='left')
            
            # Saison explizit setzen
            merged_df['SEASON'] = season
            
            all_seasons_data.append(merged_df)
            
        except Exception as e:
            print(f"FEHLER bei Saison {season}: {e}")
        
        # Wichtig: Pause für die API
        time.sleep(2)

    # -----------------------------
    # Finalisierung
    # -----------------------------
    if all_seasons_data:
        final_df = pd.concat(all_seasons_data, ignore_index=True)

        # Datentypen optimieren (String zu Float wo nötig)
        numeric_cols = ['LOC_X', 'LOC_Y', 'SHOT_DISTANCE', 'PLAYER_SEASON_FG_PCT', 'PLAYER_SEASON_3P_PCT']
        for col in numeric_cols:
            if col in final_df.columns:
                final_df[col] = pd.to_numeric(final_df[col], errors='coerce')

        # Speichern
        final_df.to_parquet(filename, index=False)
        print(f"\nFERTIG! Datei gespeichert unter: {filename}")
        print(f"Gesamtanzahl Datensätze: {len(final_df)}")
        print("Enthaltene Skill-Spalten: PLAYER_SEASON_FG_PCT, PLAYER_SEASON_3P_PCT")
        
        return final_df
    else:
        print("Keine Daten gesammelt.")
        return None

# -----------------------------
# Ausführen
# -----------------------------
if __name__ == "__main__":
    # Dies erstellt die Datei mit Daten von 2019-20 bis 2024-25
    df = create_comprehensive_nba_dataset(start_year=2019, end_year=2025)

# Load Play-by-Play Data

Defines helper functions for fetching play-by-play data with retries and exponential backoff, then constructs and saves a season-by-season play-by-play Parquet dataset.

In [ ]:
# -----------------------------
# Stabile Konfiguration
# -----------------------------
CUSTOM_HEADERS = {
    'Host': 'stats.nba.com',
    'Connection': 'keep-alive',
    'Pragma': 'no-cache',
    'Cache-Control': 'no-cache',
    'Accept': 'application/json, text/plain, */*',
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/122.0.0.0 Safari/537.36'
    ),
    'Origin': 'https://www.nba.com',
    'Sec-Fetch-Site': 'same-site',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Dest': 'empty',
    'Referer': 'https://www.nba.com/',
    'Accept-Language': 'en-US,en;q=0.9',
}



def fetch_play_by_play_with_retry(game_id, retries=5):
    """
    PlayByPlayV3 mit Retry + Exponential Backoff
    """
    for attempt in range(1, retries + 1):
        try:
            pbp = playbyplayv3.PlayByPlayV3(
                game_id=game_id,
                headers=CUSTOM_HEADERS,
                timeout=60
            ).get_data_frames()[0]

            return pbp  # Erfolg!

        except Exception as e:
            wait = 2 ** attempt
            print(f"⚠️ Fehler bei GameID {game_id} (Versuch {attempt}/{retries}): {e}")
            print(f"   → Warte {wait} Sekunden...")
            time.sleep(wait)

    # Nach allen Versuchen fehlgeschlagen
    return None



def create_nba_playbyplay_dataset(start_year=2019, end_year=2025, filename="nba_pbp_data_full.parquet"):
    """
    Lädt Play-by-Play Daten Saison für Saison, mit robustem Retry-System.
    """

    all_pbp_data = []
    failed_game_ids = []      # <- NEU: Speicher fehlerhafte IDs
    seasons = [f"{y}-{str(y+1)[-2:]}" for y in range(start_year, end_year)]

    print(f"🚀 Starte PBP Download für Saisons: {seasons[0]} bis {seasons[-1]}")
    print("Hinweis: Dieser Prozess dauert länger, da jedes Spiel einzeln geladen wird.")
    print("-" * 60)

    # -----------------------------
    # Schleife über die Saisons
    # -----------------------------
    for season in seasons:
        print(f"\n📅 [Phase 1] Lade GameIDs für Saison {season}...")

        try:
            game_finder = leaguegamefinder.LeagueGameFinder(
                season_nullable=season,
                league_id_nullable='00',
                season_type_nullable='Regular Season',
                headers=CUSTOM_HEADERS
            )
            games = game_finder.get_data_frames()[0]
            game_ids = games['GAME_ID'].unique()

            print(f"   -> {len(game_ids)} Spiele gefunden. Beginne Download...")

        except Exception as e:
            print(f"❌ Kritischer Fehler beim Laden der Spielübersicht für Saison {season}: {e}")
            continue

        season_pbp_list = []

        # -----------------------------
        # Schleife über alle Spiele
        # -----------------------------
        for i, game_id in enumerate(game_ids):

            pbp = fetch_play_by_play_with_retry(game_id, retries=5)

            if pbp is None:
                print(f"❌ GameID {game_id} endgültig fehlgeschlagen. Speichere in Fehlerliste.")
                failed_game_ids.append({"season": season, "game_id": game_id})
                continue

            # Hilfsspalten
            pbp['SEASON'] = season
            pbp['GAME_ID'] = game_id

            season_pbp_list.append(pbp)

            # Fortschrittsanzeige
            if (i + 1) % 100 == 0:
                print(f"      ... {i + 1}/{len(game_ids)} Spiele geladen")

            # Pause zur Vermeidung von Ratelimits
            time.sleep(1.2)

        # Saison finalisieren
        if season_pbp_list:
            season_df = pd.concat(season_pbp_list, ignore_index=True)
            all_pbp_data.append(season_df)
            print(f"✅ Saison {season} abgeschlossen: {len(season_df)} Zeilen.")

    # -----------------------------
    # Finalisierung
    # -----------------------------
    if all_pbp_data:
        print("\n💾 Erstelle finale Datei...")

        final_df = pd.concat(all_pbp_data, ignore_index=True)

        # Datentyp-Korrekturen
        numeric_cols = ['PERIOD', 'PC_TIME', 'WC_TIME']
        for col in numeric_cols:
            if col in final_df.columns:
                final_df[col] = pd.to_numeric(final_df[col], errors='coerce')

        # Datei speichern
        final_df.to_parquet(filename, index=False)

        print(f"\n🎉 FERTIG! Datei gespeichert unter: {filename}")
        print(f"Gesamtanzahl PBP-Events: {len(final_df)}")

    else:
        print("❌ Keine Daten gesammelt.")
        final_df = None

    # -----------------------------
    # Fehlerliste speichern
    # -----------------------------
    if failed_game_ids:
        with open("failed_game_ids.json", "w") as f:
            json.dump(failed_game_ids, f, indent=4)

        print(f"\n⚠️ {len(failed_game_ids)} Spiele konnten NICHT geladen werden.")
        print("   → gespeichert in failed_game_ids.json")

    else:
        print("\n✅ Keine fehlgeschlagenen Spiele. Alles erfolgreich geladen!")

    return final_df



# -----------------------------
# Ausführen
# ---------------ac--------------
if __name__ == "__main__":
    df = create_nba_playbyplay_dataset(start_year=2019, end_year=2025)


🚀 Starte PBP Download für Saisons: 2019-20 bis 2024-25
Hinweis: Dieser Prozess dauert länger, da jedes Spiel einzeln geladen wird.
------------------------------------------------------------

📅 [Phase 1] Lade GameIDs für Saison 2019-20...
   -> 1059 Spiele gefunden. Beginne Download...
      ... 100/1059 Spiele geladen
      ... 200/1059 Spiele geladen
      ... 300/1059 Spiele geladen
      ... 400/1059 Spiele geladen
      ... 500/1059 Spiele geladen
      ... 600/1059 Spiele geladen
⚠️ Fehler bei GameID 0021900456 (Versuch 1/5): HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=60)
   → Warte 2 Sekunden...
⚠️ Fehler bei GameID 0021900456 (Versuch 2/5): HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=60)
   → Warte 4 Sekunden...
⚠️ Fehler bei GameID 0021900456 (Versuch 3/5): HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=60)
   → Warte 8 Sekunden...
⚠️ Fehler bei GameID 0021900456

# Load all Player-Info Data

Reads the shotchart Parquet to get unique player IDs, queries `commonplayerinfo` for each player, logs failures, and writes successful profiles to CSV.

In [ ]:
import pandas as pd
import time
import os
import csv
import sys
from datetime import datetime
from nba_api.stats.endpoints import commonplayerinfo
from tqdm.notebook import tqdm

sys.path.append('../')

# --- KONFIGURATION ---
OUTPUT_DIR = '../data/raw/nba_player_data/'
PROFILE_FILE = os.path.join(OUTPUT_DIR, 'all_player_profiles_from_shots.csv') # Neuer Dateiname zur Sicherheit
FAILED_LOG_FILE = os.path.join(OUTPUT_DIR, 'failed_requests_shots.csv')

# Pfad zu deinen Shotchart-Daten (ANPASSEN!)
SHOTCHART_FILE = '../data/raw/nba_raw_shot_data.parquet' 

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Fehler-Log Datei initialisieren ---
if not os.path.exists(FAILED_LOG_FILE):
    with open(FAILED_LOG_FILE, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['timestamp', 'player_id', 'error_message'])

def log_failure(player_id, error_message):
    with open(FAILED_LOG_FILE, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        writer.writerow([timestamp, player_id, error_message])

# ==========================================
# SCHRITT 1: IDs aus Shotchart-Daten holen
# ==========================================
print("SCHRITT 1/2: Lese Spieler-IDs aus Shotchart-Daten...")

# Falls du die Daten laden musst (ansonsten diese Zeile auskommentieren, falls df_shots schon existiert)
# Achte darauf, dass der Separator stimmt (oft ',' oder ';')
try:
    df_shots = pd.read_parquet(SHOTCHART_FILE) 
    
    # Prüfen, wie die ID-Spalte heißt (meistens 'PLAYER_ID')
    id_col = 'PLAYER_ID' 
    if id_col not in df_shots.columns:
        raise ValueError(f"Spalte '{id_col}' nicht gefunden! Vorhandene Spalten: {df_shots.columns}")

    # Nur die einzigartigen IDs holen
    player_id_list = df_shots[id_col].unique().tolist()
    player_id_list = sorted(player_id_list) # Sortieren für Ordnung
    
    print(f"{len(player_id_list)} eindeutige Spieler in den Shotcharts gefunden.")

except Exception as e:
    print(f"Kritischer Fehler beim Laden der Shotcharts: {e}")
    player_id_list = [] # Damit das Skript unten nicht crasht


# ==========================================
# SCHRITT 2: Profile laden (wie zuvor)
# ==========================================
if player_id_list:
    print("\n SCHRITT 2/2: Lade detaillierte Infos für diese Spieler...")

    player_profiles = []
    error_count = 0

    for player_id in tqdm(player_id_list, desc="Profile downloaden"):
        try:
            # API Abfrage
            info = commonplayerinfo.CommonPlayerInfo(player_id=player_id, timeout=1)
            df_info = info.common_player_info.get_data_frame()
            player_profiles.append(df_info)
            
            time.sleep(0.6) # Wichtig!
            
        except Exception as e:
            error_msg = str(e)
            print(f"Fehler bei ID {player_id}: {error_msg}")
            log_failure(player_id, error_msg)
            error_count += 1

    # ==========================================
    # SPEICHERN
    # ==========================================
    print("\n Speichere Daten...")

    if player_profiles:
        final_profiles = pd.concat(player_profiles, ignore_index=True)
        
        # Sicherstellen, dass die ID-Spalte einheitlich heißt
        if 'PERSON_ID' in final_profiles.columns:
            final_profiles.rename(columns={'PERSON_ID': 'PLAYER_ID'}, inplace=True)

        final_profiles.to_csv(PROFILE_FILE, index=False)
        print(f"Profile gespeichert: {PROFILE_FILE}")
    else:
        print("Keine Profile erfolgreich geladen.")

    if error_count > 0:
        print(f"{error_count} Fehler im Log: {FAILED_LOG_FILE}")
else:
    print("Keine Spieler-IDs zum Verarbeiten gefunden.")

🔍 SCHRITT 1/2: Lese Spieler-IDs aus Shotchart-Daten...
➡️ 1121 eindeutige Spieler in den Shotcharts gefunden.

🔍 SCHRITT 2/2: Lade detaillierte Infos für diese Spieler...


Profile downloaden:   0%|          | 0/1121 [00:00<?, ?it/s]

❌ Fehler bei ID 203124: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=1)
❌ Fehler bei ID 1626187: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=1)
❌ Fehler bei ID 1629121: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=1)
❌ Fehler bei ID 1629244: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=1)
❌ Fehler bei ID 1629738: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=1)
❌ Fehler bei ID 1629739: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=1)
❌ Fehler bei ID 1629740: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=1)
❌ Fehler bei ID 1629741: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=1)
❌ Fehler bei ID 1629742: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=1)
❌ 

Retries failed `commonplayerinfo` requests from the error log, appends any newly fetched profiles to the main CSV, and updates the failure log.

In [ ]:
import pandas as pd
import time
import os
import csv
import sys
from datetime import datetime
from nba_api.stats.endpoints import commonplayerinfo
from tqdm.notebook import tqdm

# --- KONFIGURATION (Identisch lassen) ---
OUTPUT_DIR = '../data/raw/nba_player_data/'
PROFILE_FILE = os.path.join(OUTPUT_DIR, 'all_player_profiles_from_shots.csv')
FAILED_LOG_FILE = os.path.join(OUTPUT_DIR, 'failed_requests_shots.csv')

# ==========================================
# SCHRITT 1: Fehlgeschlagene IDs laden
# ==========================================
print("Lese fehlgeschlagene IDs...")

if not os.path.exists(FAILED_LOG_FILE):
    print("Keine Fehler-Datei gefunden. Alles scheint erledigt zu sein!")
    retry_ids = []
else:
    try:
        df_failed = pd.read_csv(FAILED_LOG_FILE)
        if df_failed.empty:
            print("Fehler-Log ist leer.")
            retry_ids = []
        else:
            # Wir holen uns nur die einzigartigen IDs aus dem Fehler-Log
            retry_ids = df_failed['player_id'].unique().tolist()
            print(f"{len(retry_ids)} IDs zum erneuten Versuch gefunden.")
    except Exception as e:
        print(f"Fehler beim Lesen des Error-Logs: {e}")
        retry_ids = []

# ==========================================
# SCHRITT 2: Retry Loop
# ==========================================
if retry_ids:
    print(f"\n Starte Retry für {len(retry_ids)} Spieler...")

    new_profiles = []
    still_failed = [] # Liste für IDs, die wieder fehlschlagen

    for player_id in tqdm(retry_ids, desc="Retrying"):
        try:
            # API Abfrage
            info = commonplayerinfo.CommonPlayerInfo(player_id=player_id, timeout=1) # Timeout leicht erhöht für Retries
            df_info = info.common_player_info.get_data_frame()
            
            new_profiles.append(df_info)
            print(f"✅ Retry erfolgreich für ID {player_id}")
            # WICHTIG: Wenn es geklappt hat, NICHT in die still_failed Liste aufnehmen
            
            time.sleep(0.6) 
            
        except Exception as e:
            error_msg = str(e)
            print(f"❌ Erneuter Fehler bei ID {player_id}: {error_msg}")
            
            # Timestamp für das neue Log
            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            still_failed.append([ts, player_id, error_msg])

    # ==========================================
    # SCHRITT 3: Daten updaten (Append)
    # ==========================================
    
    # 1. Erfolgreiche Profile ANHÄNGEN
    if new_profiles:
        df_new = pd.concat(new_profiles, ignore_index=True)
        
        if 'PERSON_ID' in df_new.columns:
            df_new.rename(columns={'PERSON_ID': 'PLAYER_ID'}, inplace=True)

        # Prüfen, ob die Hauptdatei existiert (für den Header)
        file_exists = os.path.exists(PROFILE_FILE)
        
        # mode='a' bedeutet APPEND (anhängen). header=not file_exists schreibt Header nur, wenn Datei neu ist.
        df_new.to_csv(PROFILE_FILE, mode='a', header=not file_exists, index=False)
        print(f"\n {len(df_new)} Profile erfolgreich zur Datei hinzugefügt!")
    else:
        print("\n Keine Profile beim Retry erfolgreich geladen.")

    # 2. Fehler-Log ÜBERSCHREIBEN (Clean up)
    # Wir überschreiben die alte Error-File. Wenn still_failed leer ist, ist die Datei danach leer (gut so).
    # Wenn noch Fehler da sind, stehen nur noch die drin, die wirklich kaputt sind.
    
    print("Aktualisiere Fehler-Log...")
    with open(FAILED_LOG_FILE, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['timestamp', 'player_id', 'error_message']) 
        if still_failed:
            writer.writerows(still_failed)
            print(f"Es verbleiben {len(still_failed)} Fehler im Log.")
        else:
            print("Fehler-Log ist nun sauber (alle Retries erfolgreich).")

else:
    print("Nichts zu tun.")

Lese fehlgeschlagene IDs...
419 IDs zum erneuten Versuch gefunden.

 Starte Retry für 419 Spieler...


Retrying:   0%|          | 0/419 [00:00<?, ?it/s]

✅ Retry erfolgreich für ID 203124
✅ Retry erfolgreich für ID 1626187
✅ Retry erfolgreich für ID 1629121
✅ Retry erfolgreich für ID 1629244
✅ Retry erfolgreich für ID 1629738
✅ Retry erfolgreich für ID 1629739
✅ Retry erfolgreich für ID 1629740
✅ Retry erfolgreich für ID 1629741
✅ Retry erfolgreich für ID 1629742
✅ Retry erfolgreich für ID 1629743
✅ Retry erfolgreich für ID 1629744
✅ Retry erfolgreich für ID 1629745
✅ Retry erfolgreich für ID 1629750
✅ Retry erfolgreich für ID 1629751
✅ Retry erfolgreich für ID 1629752
✅ Retry erfolgreich für ID 1629755
✅ Retry erfolgreich für ID 1629760
✅ Retry erfolgreich für ID 1629783
✅ Retry erfolgreich für ID 1629833
✅ Retry erfolgreich für ID 1629873
✅ Retry erfolgreich für ID 1629875
✅ Retry erfolgreich für ID 1629958
✅ Retry erfolgreich für ID 1629962
✅ Retry erfolgreich für ID 1630162
✅ Retry erfolgreich für ID 1630163
✅ Retry erfolgreich für ID 1630164
✅ Retry erfolgreich für ID 1630165
✅ Retry erfolgreich für ID 1630166
✅ Retry erfolgreich f